In [ ]:
Subir al repo un readme  explicando que hay y como se usa crear subcarpeta
codigos ABB B y B+ con códigos documentados
comparar los tiempos de ejecucion (comparar tiempos de busqueda)

In [3]:
##DATOS
import random
nombres = [
    "Sofía", "Mateo", "Valentina", "Sebastián", "Camila",
    "Nicolás", "Isabella", "Alejandro", "Daniela", "Diego",
    "Mariana", "Gabriel", "Valeria", "Emilio", "Luciana"
]
estudiantes = []
for i in range(10000):
  estudiantes.append({"id":i, "nombre":random.choice(nombres), "promedio": random.randint(0,10)})

In [14]:
def buscar_por_id_en_lista(lista, id_a_buscar):
    for item in lista:
        if item.get("id") == id_a_buscar:
            return item
    return None
buscar_por_id_en_lista(estudiantes, 18)

{'id': 18, 'nombre': 'Isabella', 'promedio': 7}

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
-------------------------------------------------------


In [31]:
#Árbol ABB
class NodoABB():
  def __init__(self, valor):
    self.valor = valor
    self.izquierda = None
    self.derecha = None
    self.padre = None

class ABB():
  def __init__(self):
    self.root = None

  def buscarPorID(self, valor):
    if self.root == None:
      return None
    else:
      nodo_actual = self.root
      while nodo_actual:
        if nodo_actual.valor["id"] == valor:
          return nodo_actual.valor
        elif valor < nodo_actual.valor["id"]:
          nodo_actual = nodo_actual.izquierda
        else:
          nodo_actual = nodo_actual.derecha
      return None

  def insertar(self, valor): # valor = {id, nombre, promedio}
    nuevo_nodo = NodoABB(valor)
    if self.root == None:
      self.root = nuevo_nodo
    else:
      nodo_actual = self.root
      while nodo_actual:
        if valor["id"] < nodo_actual.valor["id"]:
          if nodo_actual.izquierda == None:
            nuevo_nodo = NodoABB(valor)
            nodo_actual.izquierda = nuevo_nodo
            nuevo_nodo.padre = nodo_actual
            break
          else:
            nodo_actual= nodo_actual.izquierda
        elif valor["id"] > nodo_actual.valor["id"]:
          if nodo_actual.derecha == None:
            nuevo_nodo = NodoABB(valor)
            nodo_actual.derecha = nuevo_nodo
            nuevo_nodo.padre = nodo_actual
            break
          else:
            nodo_actual= nodo_actual.derecha

In [ ]:
arbol= ABB()
for estudiante in estudiantes:
  arbol.insertar(estudiante)

{'id': 18, 'nombre': 'Emilio', 'promedio': 7}

In [ ]:
arbol.buscarPorID(18)

{'id': 18, 'nombre': 'Emilio', 'promedio': 7}

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
-------------------------------------------------------


In [12]:
class NodoHoja:
    def __init__(self, estudiantes):
        self.estudiantes = sorted(estudiantes, key=lambda x: x['id'])
        self.siguiente = None

class NodoInterno:
    def __init__(self, id_guia):
        self.id_guia = id_guia
        self.hijos = []

class ArbolBPlus:
    def __init__(self, datos_estudiantes):
        # Al instanciar la clase, se construye el árbol automáticamente
        self.raiz = self._construir_estatico(datos_estudiantes)

    def _construir_estatico(self, lista):
        est_ord = sorted(lista, key=lambda x: x['id'])
        n = len(est_ord)

        # 1. Raíz
        punto_medio = n // 2
        raiz = NodoInterno(est_ord[punto_medio]['id'])

        bloque_izq = est_ord[:punto_medio]
        bloque_der = est_ord[punto_medio:]

        # 2. Nivel Intermedio (6 hijos)
        for i in range(6):
            if i < 3:
                sub = bloque_izq[i * len(bloque_izq)//3 : (i+1) * len(bloque_izq)//3]
            else:
                idx = i - 3
                sub = bloque_der[idx * len(bloque_der)//3 : (idx+1) * len(bloque_der)//3]

            id_guia = sub[len(sub)//2]['id'] if sub else 0
            n_intermedio = NodoInterno(id_guia)
            raiz.hijos.append(n_intermedio)

            # 3. Nivel Hojas (6 por intermedio)
            for j in range(6):
                datos_h = sub[j * len(sub)//6 : (j+1) * len(sub)//6]
                n_intermedio.hijos.append(NodoHoja(datos_h))

        return raiz

    def buscar_por_id(self, target_id):
        """Busca un estudiante navegando por los 3 niveles."""

        # Nivel 1: Raíz
        # Decidimos si ir a la primera mitad (hijos 0-2) o segunda mitad (3-5)
        if target_id < self.raiz.id_guia:
            candidatos = self.raiz.hijos[:3]
        else:
            candidatos = self.raiz.hijos[3:]

        # Nivel 2: Nodo Intermedio
        # Como no tenemos 5 claves para 6 hijos, buscamos en el sub-bloque de candidatos
        # basándonos en la guía de cada uno.
        objetivo_intermedio = candidatos[-1] # Por defecto el último del bloque
        for i in range(len(candidatos) - 1):
            if target_id < candidatos[i+1].id_guia:
                objetivo_intermedio = candidatos[i]
                break


        # Nivel 3: Hojas
        # Misma lógica para encontrar la hoja correcta entre las 6 disponibles
        hoja_objetivo = objetivo_intermedio.hijos[-1]
        for i in range(len(objetivo_intermedio.hijos) - 1):
            # Miramos el primer ID de la siguiente hoja para saber si el target está en la actual
            siguiente_hoja = objetivo_intermedio.hijos[i+1]
            if siguiente_hoja.estudiantes and target_id < siguiente_hoja.estudiantes[0]['id']:
                hoja_objetivo = objetivo_intermedio.hijos[i]
                break

        # Búsqueda final dentro de la hoja
        for estudiante in hoja_objetivo.estudiantes:
            if estudiante['id'] == target_id:
                return estudiante

        return None

# --- EJEMPLO DE USO ---

mi_arbol = ArbolBPlus(estudiantes)

# Buscar un estudiante específico
resultado = mi_arbol.buscar_por_id(9805)




In [17]:
import time
import random

ArbolBPlus = ArbolBPlus(estudiantes)
# Configuracion del test
iteraciones = 100000
ids_a_buscar = [random.randint(1, 10000) for _ in range(iteraciones)]

# --- INICIO DE MEDICIÓN PURA ---
t_inicio = time.process_time()

for target in ids_a_buscar:
    ArbolBPlus.buscar_por_id(target)

t_fin = time.process_time()
# --- FIN DE MEDICIÓN PURA ---

tiempo_total = t_fin - t_inicio
tiempo_promedio_por_busqueda = tiempo_total / iteraciones

print(f"📊 RESULTADOS DEL BENCHMARK")
print(f"-----------------------------------")
print(f"Tiempo total de CPU (100k búsquedas): {tiempo_total:.6f} segundos")
print(f"Tiempo promedio por ID: {tiempo_promedio_por_busqueda:.10f} segundos")

📊 RESULTADOS DEL BENCHMARK
-----------------------------------
Tiempo total de CPU (100k búsquedas): 0.927023 segundos
Tiempo promedio por ID: 0.0000092702 segundos


In [32]:
import time
import random

random.shuffle(estudiantes)
mi_abb = ABB()
for est in estudiantes:
    mi_abb.insertar(est)

# 3. Configurar la prueba
iteraciones = 100000
ids_a_buscar = [random.randint(1, 10000) for _ in range(iteraciones)]

# --- INICIO DE MEDICIÓN PURA (CPU TIME) ---
t_inicio = time.process_time()

for target in ids_a_buscar:
    buscar_por_id_en_lista(estudiantes,target)

t_fin = time.process_time()
# --- FIN DE MEDICIÓN PURA ---

tiempo_total = t_fin - t_inicio
tiempo_promedio = tiempo_total / iteraciones

print(f"📊 BENCHMARK ÁRBOL BINARIO (ABB)")
print(f"-----------------------------------")
print(f"Tiempo total de CPU: {tiempo_total:.6f} s")
print(f"Tiempo promedio por búsqueda: {tiempo_promedio:.10f} s")

📊 BENCHMARK ÁRBOL BINARIO (ABB)
-----------------------------------
Tiempo total de CPU: 0.014837 s
Tiempo promedio por búsqueda: 0.0000014837 s


In [33]:
import time
import random
# busqueda con listas
random.shuffle(estudiantes)

# 3. Configurar la prueba
iteraciones = 100000
ids_a_buscar = [random.randint(1, 10000) for _ in range(iteraciones)]

# --- INICIO DE MEDICIÓN PURA (CPU TIME) ---
t_inicio = time.process_time()

for target in ids_a_buscar:
    mi_abb.buscarPorID(target)

t_fin = time.process_time()
# --- FIN DE MEDICIÓN PURA ---

tiempo_total = t_fin - t_inicio
tiempo_promedio = tiempo_total / iteraciones

print(f"📊 BENCHMARK LISTAS")
print(f"-----------------------------------")
print(f"Tiempo total de CPU: {tiempo_total:.6f} s")
print(f"Tiempo promedio por búsqueda: {tiempo_promedio:.10f} s")

📊 BENCHMARK ÁRBOL BINARIO (ABB)
-----------------------------------
Tiempo total de CPU: 0.452750 s
Tiempo promedio por búsqueda: 0.0000045275 s
